Question 1: Install Spark and PySpark

In [1]:
import pyspark
print(pyspark.__version__)

3.5.5


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()
print('PySpark Version:'+spark.version)

PySpark Version:3.5.5


Question 2: Yellow October 2024

In [3]:
! wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-05 04:41:58--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.8.245.51, 65.8.245.50, 65.8.245.171, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.8.245.51|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet’

yellow_tripdata_202 100%[===================>]  61.36M  47.7MB/s    in 1.3s    

2025-03-05 04:41:59 (47.7 MB/s) - ‘yellow_tripdata_2024-10.parquet’ saved [64346071/64346071]



In [4]:
df_spark = spark.read.option("header", "true").parquet("/content/yellow_tripdata_2024-10.parquet")

In [6]:
df_spark.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|       18.4|  1.0|    0.5|       1.

In [7]:
df_spark.repartition(4).write.mode("overwrite").save("/content/test_par/")

Question 3: Count records

In [8]:
df_spark.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [9]:
df_spark.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [10]:
df_spark.dtypes

[('VendorID', 'int'),
 ('tpep_pickup_datetime', 'timestamp_ntz'),
 ('tpep_dropoff_datetime', 'timestamp_ntz'),
 ('passenger_count', 'bigint'),
 ('trip_distance', 'double'),
 ('RatecodeID', 'bigint'),
 ('store_and_fwd_flag', 'string'),
 ('PULocationID', 'int'),
 ('DOLocationID', 'int'),
 ('payment_type', 'bigint'),
 ('fare_amount', 'double'),
 ('extra', 'double'),
 ('mta_tax', 'double'),
 ('tip_amount', 'double'),
 ('tolls_amount', 'double'),
 ('improvement_surcharge', 'double'),
 ('total_amount', 'double'),
 ('congestion_surcharge', 'double'),
 ('Airport_fee', 'double')]

In [11]:
df_spark.select('tpep_pickup_datetime').show(5)

+--------------------+
|tpep_pickup_datetime|
+--------------------+
| 2024-10-01 00:30:44|
| 2024-10-01 00:12:20|
| 2024-10-01 00:04:46|
| 2024-10-01 00:12:10|
| 2024-10-01 00:30:22|
+--------------------+
only showing top 5 rows



In [12]:
from pyspark.sql.functions import col, to_date, lit
df_spark.select(to_date(col('tpep_pickup_datetime')).alias('pickup_date').cast("date")).show(5)

+-----------+
|pickup_date|
+-----------+
| 2024-10-01|
| 2024-10-01|
| 2024-10-01|
| 2024-10-01|
| 2024-10-01|
+-----------+
only showing top 5 rows



In [13]:
df_spark.filter(to_date(col('tpep_pickup_datetime')).cast("date") == "2024-10-15").count()

128893

In [14]:
df_sparkdate = df_spark.withColumn("PickUPDate",to_date(col('tpep_pickup_datetime')).cast("date") )

In [15]:
df_sparkdate.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PickUPDate|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|         1|                 N|         162|         246|           1|  

In [16]:
df_sparkdate.where(df_sparkdate.PickUPDate == "2024-10-15").count()

128893

Question 4: Longest trip

In [17]:
df_sparkdate = df_sparkdate.withColumn('duration_hours',df_sparkdate.tpep_dropoff_datetime -df_sparkdate.tpep_pickup_datetime)

In [18]:
df_sparkdate.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PickUPDate|      duration_hours|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+
|       2| 2024-10-01 00:30:44|  2024-10-01 00:48:26|              1|          3.0|        

In [19]:
#df_sparkdate.select(df_sparkdate.max('duration_hours')).show()
row1 = df_sparkdate.agg({"duration_hours": "max"}).collect()[0]


In [20]:
print(row1)

Row(max(duration_hours)=datetime.timedelta(days=6, seconds=67024))


In [21]:
#print(row1[max(duration_hours)])
print(row1["max(duration_hours)"])

6 days, 18:37:04


In [22]:
from pyspark.sql.functions import unix_timestamp
import pyspark.sql.functions as F
df_sparkdate = df_sparkdate.withColumn("duratinHrs",(unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime")))/3600)

In [23]:
df_sparkdate.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PickUPDate|      duration_hours|          duratinHrs|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+--------------------+
|       2| 2024-10-01 00:30:

In [24]:
df_sparkdate = df_sparkdate.withColumn("durationHrs",F.round((unix_timestamp(col("tpep_dropoff_datetime"))-unix_timestamp(col("tpep_pickup_datetime")))/3600,2))

In [25]:
df_sparkdate.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PickUPDate|      duration_hours|          duratinHrs|durationHrs|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+--------------------+-----

In [26]:
maxHrs = df_sparkdate.agg({"durationHrs": "max"}).collect()[0][0]

In [28]:
type(maxHrs)

float

In [29]:
maxHrs

162.62

In [30]:
from pyspark.sql.functions import max
df_sparkdate.select(max(df_sparkdate.durationHrs)).show()

+----------------+
|max(durationHrs)|
+----------------+
|          162.62|
+----------------+



In [31]:
df_sparkdate.select(col("trip_distance")).show(5)

+-------------+
|trip_distance|
+-------------+
|          3.0|
|          2.2|
|          2.7|
|          3.1|
|          0.0|
+-------------+
only showing top 5 rows



In [32]:
df_sparkdate.where(df_sparkdate.durationHrs == 162.62).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PickUPDate|      duration_hours|        duratinHrs|durationHrs|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+------------------+-----------

In [33]:
df_sparkdate.where(df_sparkdate.durationHrs == maxHrs).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PickUPDate|      duration_hours|        duratinHrs|durationHrs|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+------------------+-----------

In [ ]:
df_sparkdate.where(df_sparkdate.durationHrs == maxHrs).show()

In [34]:
df_sparkdate.select("trip_distance").where(df_sparkdate.durationHrs == maxHrs).show()

+-------------+
|trip_distance|
+-------------+
|        32.37|
+-------------+



Question 5: User Interface


To access the Spark UI, you would navigate to http://localhost:4040 in your web browser.

In [35]:
! wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-05 05:09:55--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 65.8.245.171, 65.8.245.51, 65.8.245.50, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|65.8.245.171|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2025-03-05 05:09:55 (14.8 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [36]:
df_taxi_zone = spark.read \
    .format("csv") \
    .option("header", "true") \
    .load("/content/taxi_zone_lookup.csv")

In [37]:
df_taxi_zone.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [38]:
df_taxi_zone.createOrReplaceTempView("temp_view")

In [39]:
result = spark.sql("SELECT * FROM temp_view")
result.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [40]:
df_final = df_sparkdate.join(df_taxi_zone,
               df_sparkdate.PULocationID == df_taxi_zone.LocationID,
               "inner")

In [41]:
df_final.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+--------------------+--------------------+-----------+----------+---------+-------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|PickUPDate|      duration_hours|          duratinHrs|durationHrs|LocationID|  Borough|               Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+------------------

In [44]:
df_final.groupBy("PULocationID").count().orderBy('count').show(1)

+------------+-----+
|PULocationID|count|
+------------+-----+
|         105|    1|
+------------+-----+
only showing top 1 row



In [47]:
loc_id = df_final.groupBy("PULocationID").count().orderBy('count').collect()[0][0]

In [49]:
df_final.select("Zone").where(df_final.PULocationID == loc_id).show()

+--------------------+
|                Zone|
+--------------------+
|Governor's Island...|
+--------------------+



In [50]:
res = spark.sql("SELECT Zone FROM temp_view where LocationID=105")

In [51]:
res.show()

+--------------------+
|                Zone|
+--------------------+
|Governor's Island...|
+--------------------+

